# Example computing connectivity in different ROIs

Matplotlib magic for interactive display

In [ ]:
%matplotlib widget

### Importing dependencies

In [ ]:
import numpy as np
from matplotlib.patches import Polygon
from matplotlib.path import Path
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from pyiconeus import open_path, Scan, Bps, Roi
from pyiconeus.models.Roi import RoiElements
from ipywidgets import *

# Scan loading

In [ ]:
scan: Scan = open_path("../tests/data/scanFile4DScan.scan")
voxels = scan.voxels
voxels.shape
norm_voxels = voxels

# Bps loading and adding to the scan

In [ ]:
bps: Bps = open_path("../tests/data/scanFile4DScan.bps")
bps.data

In [ ]:
scan.bps = bps

# Utils function to display the ROI volume

In [ ]:
def _to_row_major(arr, n_cols=3):
    """
    Normalize to shape (n, n_cols) regardless of whether the source array was
    stored as (n_cols, n) or (n, n_cols).
    """
    arr = np.asarray(arr)
    if arr.ndim != 2:
        raise ValueError(f"Expected a 2D array, got shape {arr.shape}")
    if arr.shape[0] == n_cols and arr.shape[1] != n_cols:
        return arr.T
    return arr


In [ ]:
def volume(roi: RoiElements):
    """
    Mesh volume via the divergence theorem: sum the signed volume of the
    tetrahedron formed by each face and the origin.
    """
    v = _to_row_major(roi.vertices)
    f = _to_row_major(roi.faces).astype(int)
    v0, v1, v2 = v[f[:, 0]], v[f[:, 1]], v[f[:, 2]]
    signed_vols = np.einsum('ij,ij->i', v0, np.cross(v1, v2))
    return abs(signed_vols.sum()) / 6.0


In [ ]:
def slice_contour(roi: RoiElements, coord, axis=1):
        """
        Intersect the mesh with the plane {axis-th coordinate == coord} and
        return the points of the intersection, as an (k, 2) array of
        points in the two remaining coordinates (in hull order, ready to
        pass straight to matplotlib's Polygon). Returns None if the plane
        misses the mesh (or touches it in fewer than 3 distinct points).
        """
        v = _to_row_major(roi.vertices)
        f = _to_row_major(roi.faces).astype(int)
        pts = []
        coord += 1

        for tri in f:
            tri_v = v[tri]
            c = tri_v[:, axis]
            for i in range(3):
                j = (i + 1) % 3
                c0, c1 = c[i], c[j]
                if (c0 - coord) * (c1 - coord) < 0:
                    
                    t = (coord - c0) / (c1 - c0)
                    pts.append(tri_v[i] + t * (tri_v[j] - tri_v[i]))
                elif c0 == coord:
                    pts.append(tri_v[i])

        if len(pts) < 3:
            return None

        pts = np.array(pts)
        other_axes = [a for a in range(3) if a != axis]
        pts_2d = pts[:, other_axes]
        pts_2d = np.unique(np.round(pts_2d, 6), axis=0)

        if len(pts_2d) < 3:
            return None

        return pts_2d

## Util function to apply  a 4X4 matrix transform

In [ ]:
def apply_transform(vertices, matrix):
    """
    vertices: (n, 3) array, one row per vertex
    matrix:   (4, 4) homogeneous affine, (3, 3) rotation/scale only, or (3, 4) affine
    returns:  (n, 3) transformed vertices
    """
    vertices = np.asarray(vertices, dtype=float)
    matrix = np.asarray(matrix, dtype=float)
    n = vertices.shape[0]
 
    if matrix.shape == (4, 4):
        homo = np.hstack([vertices, np.ones((n, 1))])   # (n, 4)
        transformed = homo @ matrix.T                     # (n, 4)
        return transformed[:, :3] / transformed[:, 3:4]
    else:
        raise ValueError(f"Unexpected transform shape: {matrix.shape}")


# Roi loading

In [ ]:
roi: Roi = open_path("../tests/data/Cortex.bri")

# Brain space to Voxel space transform

In [ ]:
probe2Lab = scan.get_probe_to_lab()[0]
voxel2Probe = scan.get_voxel_to_probe()
Brain2Voxel = np.linalg.inv(voxel2Probe) @ np.linalg.inv(probe2Lab) @ scan.bps.data

In [ ]:
def describe(name, M):
    # rough "scale" of a transform matrix: the norm of its linear part
    scale = np.linalg.norm(M) * np.linalg.norm(np.linalg.inv(M))
    print(f"{name}: scale~{scale:.6g}")
    print(M)
    print()

describe("bps.data", scan.bps.data)
describe("probe2Lab", probe2Lab)
describe("voxel2Probe", voxel2Probe)
describe("BrainToVoxel", Brain2Voxel)

In [ ]:
transformed_cache = {}

# 2-opt mask creation using ROI contour creation

In [ ]:
def initial_tour_nn(points: np.ndarray, start: int = 0) -> np.ndarray:
    """Greedy nearest-neighbour walk. Returns an order (array of indices)."""
    n = len(points)
    tree = cKDTree(points)
    visited = np.zeros(n, dtype=bool)
    order = [start]
    visited[start] = True
    for _ in range(n - 1):
        cur = order[-1]
        k = 2
        while True:
            k = min(k, n)
            _, idx = tree.query(points[cur], k=k)
            idx = np.atleast_1d(idx)
            cand = idx[~visited[idx]]
            if cand.size:
                nxt = int(cand[0])
                break
            if k >= n:
                nxt = int(np.where(~visited)[0][0])
                break
            k *= 2
        order.append(nxt)
        visited[nxt] = True
    return np.array(order)
 
 
def tour_length(points: np.ndarray, order: np.ndarray) -> float:
    """Distance getter from point to points in 'points' with order"""
    p = points[order]
    return float(np.sum(np.linalg.norm(p - np.roll(p, -1, axis=0), axis=1)))
 
 
def two_opt(points: np.ndarray, order: np.ndarray, max_passes: int = 200) -> np.ndarray:
    """
    Classic 2-opt local search on a cyclic tour.
    O(n^2) per pass -> fine up to a few thousand points. For larger point
    counts, restrict the inner loop to each point's spatial neighbours
    (via the same KD-tree) instead of scanning all j.
    """
    order = order.copy()
    n = len(order)
    improved = True
    passes = 0
    while improved and passes < max_passes:
        improved = False
        passes += 1
        for i in range(n - 1):
            a = order[i]
            for j in range(i + 2, n):
                b = order[i + 1]
                c = order[j]
                d = order[(j + 1) % n]
                if d == a:
                    continue
                before = (np.linalg.norm(points[a] - points[b]) +
                          np.linalg.norm(points[c] - points[d]))
                after = (np.linalg.norm(points[a] - points[c]) +
                         np.linalg.norm(points[b] - points[d]))
                if after + 1e-9 < before:
                    order[i + 1:j + 1] = order[i + 1:j + 1][::-1]
                    improved = True
    return order
 
 
def reorder_contour_points(points: np.ndarray) -> np.ndarray:
    """Unordered (N,2) points -> ordered (N,2) polygon vertices."""
    points = np.asarray(points, dtype=float)
    if len(points) < 3:
        return points
    order = initial_tour_nn(points)
    order = two_opt(points, order)
    return points[order]

# Caching the computed masks and contour

In [ ]:
X_size, Z_size = norm_voxels.shape[0], norm_voxels.shape[2]
xx, zz = np.mgrid[0:X_size, 0:Z_size]
grid_pts = np.column_stack((xx.ravel(), zz.ravel()))

roi_mask_cache = {}
roi_hull_cache = {}
roi_contour_cache = {}

for rIdx, r in enumerate(roi.list):
    if rIdx not in transformed_cache:
        transformed_cache[rIdx] = apply_transform(r.vertices, Brain2Voxel)
    r.vertices = transformed_cache[rIdx]

    for pose in range(scan.sizeY):
        roi_pose = scan.sizeY - 1 - pose
        contour = slice_contour(r, roi_pose, axis=1)
        roi_contour_cache[(rIdx, pose)] = contour
        if contour is None:
            roi_mask_cache[(rIdx, pose)] = np.zeros((X_size, Z_size), dtype=bool)
            roi_hull_cache[(rIdx, pose)] = np.zeros((X_size, Z_size))
        else:
            hull = reorder_contour_points(contour)
            print("tour length shuffled :", tour_length(contour, np.arange(len(contour))))
            print("tour length hull:", tour_length(hull, np.arange(len(hull))))

            roi_hull_cache[(rIdx, pose)] = hull
            inside = Path(hull).contains_points(grid_pts)
            roi_mask_cache[(rIdx, pose)] = inside.reshape(X_size, Z_size)

# Transposing the X and Z axis for display purpose

In [ ]:
norm_voxels = np.transpose(voxels, axes=(2, 1, 0, 3, 4, 5))

In [ ]:
norm_voxels.shape

# Interactive display

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2)
im = ax.imshow(norm_voxels[:, int(scan.sizeY / 2), :, int(scan.nTime / 2), 0, 0], cmap='gray', interpolation='bilinear')
current_state = {"pose": int(scan.sizeY / 2), "roiValue": 0, "nTime": int(scan.nTime / 2)}
poly_artist = {"poly": None, "poly2": None, "poly3": None}


def update(nTime, pose, roiValue):
    current_state["pose"], current_state["roiValue"], current_state["nTime"] = pose, roiValue, nTime

    # Display of the choosed slice
    img_slice = norm_voxels[:, pose, :, nTime, 0, 0]
    im.set_data(img_slice)
    im.set_extent([-0.5, img_slice.shape[1] - 0.5, img_slice.shape[0] - 0.5, -0.5])

    # Plot flushing
    if poly_artist["poly"] is not None:
        poly_artist["poly"] = None
    if poly_artist["poly3"] is not None:
        poly_artist["poly3"].remove()
        poly_artist["poly3"] = None
    if poly_artist["poly2"] is not None:
        poly_artist["poly2"].remove()
        poly_artist["poly2"] = None


    # Select the correct roi and its cached outputs
    selectedRoi = roi.list[roiValue]
    selectedRoi.vertices = transformed_cache[roiValue]

    contour = roi_contour_cache[(roiValue, pose)]
    if contour is not None:
        color = (selectedRoi.color, 0.4)
        contour_x = img_slice.shape[1] - 1 - contour[:, 0]
        contour_y = contour[:, 1]

        # Contour plotting
        poly_artist["poly3"] = ax.scatter(contour_x, contour_y, marker='.', c=[color])

        # Mask plotting on second ax
        mask = roi_mask_cache[(roiValue, pose)]
        poly_artist["poly"] = np.fliplr(mask.T)
        ax2.imshow(poly_artist["poly"])
        hull_pts = roi_hull_cache[(roiValue, pose)]
        hull_xy = np.column_stack((img_slice.shape[1] - 1 - hull_pts[:, 0], hull_pts[:, 1]))
        poly_artist["poly2"] = Polygon(hull_xy, closed=True, fill=False,
                                        edgecolor=color, linewidth=4)
        ax2.add_patch(poly_artist["poly2"])
    else:
        # Empty mask if no contour was found
        mask = np.zeros(roi_mask_cache[(roiValue, pose)].shape)
        poly_artist["poly"] = np.fliplr(mask.T)
        ax2.imshow(poly_artist["poly"])

    # Draw call
    fig.canvas.draw_idle()


# Interactive widget with sliders and a dropdown
interact(
    update,
    nTime=widgets.IntSlider(value=int(scan.nTime / 2), min=0, max=scan.nTime - 1, step=1),
    pose=widgets.IntSlider(value=int(scan.sizeY / 2), min=0, max=scan.sizeY - 1, step=1),
    roiValue=widgets.Dropdown(
        options = [(roi.list[i].name, i) for i in range(len(roi.list))],
        value = 0,
        description = "ROI: ",
    )
)


# getSignal function

### Returns the extracted time signal in the specific ROI

In [ ]:
def getSignal(roiValue: int, pose: int):
    mask = roi_mask_cache[(roiValue, pose)]
    data_over_time = norm_voxels[:, pose, :, :, 0, 0]
    data_over_time = np.transpose(data_over_time, axes=(1, 0, 2))
    return (data_over_time * mask[:, :, None]).sum(axis=(0, 1)) / (mask.sum() if mask.sum() != 0 else 1)

# Compute the volumic time of the acquisition (Time per full volume acquisition)

In [ ]:
measuredTimeVolumic = np.zeros(scan.nTime)
for i in range(scan.nTime):
    measuredTimeVolumic[i] = max(np.max(scan.measuredTimes[i*scan.sizeY:i*scan.sizeY+scan.sizeY]), measuredTimeVolumic[i])

# Display the previously selected pose and ROI extracted signal

Values from the previous interactive display

In [ ]:
pose = current_state["pose"]
roiValue = current_state["roiValue"]

fig_time, ax_time = plt.subplots(1, scan.voxels.shape[1])
for i in range(scan.voxels.shape[1]):
    selectedRoi = roi.list[roiValue]
    if roiValue not in transformed_cache:
        transformed_cache[roiValue] = apply_transform(selectedRoi.vertices, Brain2Voxel)
    selectedRoi.vertices = transformed_cache[roiValue]

    roiSignal = getSignal(roiValue, i)


    ax_time[i].plot(measuredTimeVolumic, roiSignal, linewidth=1)
    ax_time[i].set_xlabel('Time (s)')
    ax_time[i].set_ylabel('Mean Doppler')
    ax_time[i].set_title(f'ROI: {selectedRoi.name}, pose={i}')
fig_time.canvas.draw_idle()

# Compute the connectivity between all the ROIs in the bri file


### The correlation uses the pearson correlation

In [ ]:
def computeConnectivity(roi: Roi, npose: int) -> np.ndarray:
    volumicSignals = []
    roiNames = []
    for i in range(len(roi.list)):
        vol = []
        for j in range(npose):
            signal = getSignal(i, j)
            if not np.allclose(signal, np.zeros(signal.shape)):
                vol.append(signal)
        if len(vol) > 0:
            vol = np.stack(vol, axis=1)
            volumicSignals.append(vol.mean(axis=1))
            roiNames.append(roi.list[i].name)
    connectivity = np.corrcoef(volumicSignals).round(decimals=5)
    return connectivity, roiNames, volumicSignals

In [ ]:
# Compute the mean of every slice to have the volumic mean doppler by the time
# Connectivity = Pearson correlation between two ROIs ((L), (R))

connectivity, roiNames, volumicSignals = computeConnectivity(roi, norm_voxels.shape[1])

In [ ]:
connectivity.shape

## Display of the correlation heatmap

In [ ]:
fig_connectivity, ax_connectivity = plt.subplots()
im_con = ax_connectivity.imshow(connectivity, cmap='rainbow', vmin=-1.0, vmax=1.0)

cbar = ax_connectivity.figure.colorbar(im_con, ax=ax_connectivity)
cbar.ax.set_ylabel('Connectivity', rotation=-90, va="bottom")
cbar.set_ticks(np.arange(-1, 1, 0.2))

ax_connectivity.set_xticks(range(len(roiNames)), labels=roiNames, rotation=15, rotation_mode="anchor")
ax_connectivity.set_yticks(range(len(roiNames)), labels=roiNames)

for i in range(len(roiNames)):
    for j in range(len(roiNames)):
        text = ax_connectivity.text(j, i, connectivity[i, j], ha='center', va='center', color='w')

ax_connectivity.set_title("Connectivity of different Regions of Interest")
plt.show()

In [ ]:
fig_average_doppler, ax_average_doppler = plt.subplots()
for roi_name, volume in zip(roiNames, volumicSignals):
    ax_average_doppler.plot(volume, label=roi_name)

ax_average_doppler.legend()
plt.show()